In [1]:
import os, time, json, httpx
from dotenv import load_dotenv

load_dotenv()
LLM = httpx.Client(base_url="https://api.cerebras.ai/v1", timeout=30,
                   headers={"Authorization": f"Bearer {os.environ['CEREBUS']}"})
MODEL = "gpt-oss-120b"

def timeit(prompt, system="Respond shortly.", stream=True):
    body = {"model": MODEL, "stream": stream,
            "messages": [{"role": "system", "content": system},
                         {"role": "user", "content": prompt}]}
    t0 = time.perf_counter()
    if not stream:
        r = LLM.post("/chat/completions", json=body).json()
        total = time.perf_counter() - t0
        text = r["choices"][0]["message"]["content"]
        print(f"{text}\n\ntotal {total:.3f}s")
        return total
    first, text = None, ""
    with LLM.stream("POST", "/chat/completions", json=body) as r:
        for line in r.iter_lines():
            if not line.startswith("data: ") or line == "data: [DONE]":
                continue
            d = json.loads(line[6:])["choices"][0]["delta"].get("content")
            if d:
                first = first or time.perf_counter() - t0   # time to first token
                text += d
    total = time.perf_counter() - t0
    print(f"{text}\n\nfirst token {first:.3f}s | total {total:.3f}s")
    return first, total

In [2]:
PROMPT = "How far away is the moon?"

timeit(PROMPT)

The Moon is about 384,400 km (≈238,900 miles) from Earth on average—its distance varies between roughly 363,300 km and 405,500 km due to its elliptical orbit.

first token 0.322s | total 0.349s


(0.3219237909652293, 0.349455458112061)

In [3]:
# same prompt, N times - warm connection, median is the honest number
import statistics
runs = [timeit(PROMPT)[0] for _ in range(3)]
print(f"\nmedian first token {statistics.median(runs):.3f}s")

The Moon is about 384,400 kilometers (≈ 238,900 miles) from Earth on average, though its distance varies by a few thousand kilometers due to its elliptical orbit.

first token 0.277s | total 0.296s
About 384,400 km (≈238,855 mi) on average—the distance varies from ~363,300 km at perigee to ~405,500 km at apogee.

first token 0.307s | total 0.314s
The Moon is about 384,400 km (≈ 238,900 mi) from Earth on average—ranging roughly from 363,300 km to 405,500 km due to its elliptical orbit.

first token 0.499s | total 0.524s

median first token 0.307s
